# backward-on-scalar-loss — faded example 1: Complete the scalar reduction before backward()

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-on-scalar-loss`. Running the beacon reports progress on the `PyTorch: backward()` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: backward()` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-on-scalar-loss`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-on-scalar-loss"
DD_SUBTOPIC = "PyTorch: backward()"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

`.backward()` requires a scalar. A per-sample loss vector must be reduced first. Using `.mean()` divides the summed gradient by N; the reduction is the only thing standing between the vector loss and a valid `.backward()` call.

## Faded exercise 1

### Faded — reduce the per-sample loss so backward() works

Implement `faded_mean_reduce(w, x, y)`. The per-sample squared-error loss is already computed for you. **Complete the one missing step: reduce the per-sample loss vector to a scalar with `.mean()`** so that `.backward()` can run and populate `w.grad`. Return `(scalar_loss, w.grad)`.

**Fill in:** Reduce the per-sample loss vector to a scalar by taking its mean.

In [ ]:
def faded_mean_reduce(w, x, y):
    per_sample_loss = (w * x - y) ** 2
    scalar_loss = per_sample_loss.mean()
    scalar_loss.backward()
    return scalar_loss, w.grad

t.manual_seed(0)
w = t.tensor([1.2], requires_grad=True)
x = t.tensor([1.0, 2.0, 3.0, 4.0])
y = t.tensor([0.0, 1.0, 1.0, 2.0])
print(faded_mean_reduce(w, x, y))


def _test():
    w = t.tensor([1.2], requires_grad=True)
    x = t.tensor([1.0, 2.0, 3.0, 4.0])
    y = t.tensor([0.0, 1.0, 1.0, 2.0])
    scalar_loss, grad = faded_mean_reduce(w, x, y)
    assert scalar_loss.dim() == 0, 'loss must be a scalar'
    n = x.numel()
    expected_loss = ((1.2 * x - y) ** 2).mean()
    assert t.allclose(scalar_loss.detach(), expected_loss), 'scalar loss wrong'
    expected_grad = (2 / n) * ((1.2 * x - y) * x).sum()
    assert t.allclose(grad, expected_grad.reshape(1)), 'w.grad wrong (mean reduction expected)'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def faded_mean_reduce(w, x, y):
    per_sample_loss = (w * x - y) ** 2
    scalar_loss = per_sample_loss.mean()
    scalar_loss.backward()
    return scalar_loss, w.grad

t.manual_seed(0)
w = t.tensor([1.2], requires_grad=True)
x = t.tensor([1.0, 2.0, 3.0, 4.0])
y = t.tensor([0.0, 1.0, 1.0, 2.0])
print(faded_mean_reduce(w, x, y))
```
</details>